Importing the necessary functions and loading the model 

In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


In [2]:
import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

In [5]:
import importlib 
import eval_utils
importlib.reload(eval_utils)
from eval_utils import load_model
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-29/no_encode_intensity_concat_comp_concat_neg_mask_mp_20")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


  0%|          | 0/9046 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))
/home/gridsan/tmackey/.conda/envs/cdvae/

CrystDataModule(self.datasets={'train': {'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy train', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/train.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}, 'val': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy val', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/val.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}], 'test': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy test', 'path': '/home/gridsan/tmackey/cdvae/data/mp_20/test.csv', 'prop': 'formation_energy_per_atom', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}]}, self

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [154]:
model.to('cuda:0')

for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)

batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

_, _, z = model.encode(batch, xrd_int, xrd_loc, atom_spec)

First stage: show that the code works for a single crystal with stripped-down code 

In [162]:
num_atoms = batch.num_atoms[[0]].to('cuda:0')
atom_types = atom_spec[[0]].to('cuda:0')
z = z[[0]].to('cuda:0')

In [164]:
#### inputs 
gt_num_atoms = num_atoms
gt_elements_input = atom_types
num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(
            z, gt_num_atoms, gt_elements=gt_elements_input)
composition_per_atom = composition_per_atom.to('cuda:0') # this can be removed in native code 
num_atoms = batch.num_atoms[[0]].to('cuda:0')  #this can be removed in native code 

# Create a range tensor and repeat it as before
range_tensor = torch.arange(len(num_atoms), device='cuda:0')
crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

# Convert atom_types into a mask
atom_mask = atom_types != 0

# For each unique crystal_id, get its corresponding indices in composition_per_atom
unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)

composition_per_atom = composition_per_atom + 1

start_idx = 0
for u_id, count in zip(unique_crystal_ids, counts):
    relevant_elements = atom_types[u_id][atom_mask[u_id]]
    mask = torch.ones_like(composition_per_atom[start_idx])
    mask *= (-10**6)
    mask[relevant_elements-1] = 1

    # Create additive mask
    additive_mask_for_normalization = mask 
    additive_mask_for_normalization[relevant_elements-1] *= 0.0001

    # Apply masks to the relevant segment of composition_per_atom
    composition_per_atom[start_idx:start_idx+count] *= mask
    composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

    # Update start index for next iteration
    start_idx += count

In [165]:
self = model

In [166]:
batch.num_atoms = batch.num_atoms.to('cuda:0')

In [167]:
noise_level = torch.randint(0, self.sigmas.size(0),
                                    (batch.num_atoms.size(0),),
                                    device=self.device)
used_sigmas_per_atom = self.sigmas[noise_level].repeat_interleave(
    batch.num_atoms, dim=0)

type_noise_level = torch.randint(0, self.type_sigmas.size(0),
                                (batch.num_atoms.size(0),),
                                device=self.device)
used_type_sigmas_per_atom = (
    self.type_sigmas[type_noise_level].repeat_interleave(
        batch.num_atoms, dim=0))

In [168]:
pred_composition_per_atom = composition_per_atom
pred_composition_probs = F.softmax(
            pred_composition_per_atom.detach(), dim=-1)

In [169]:
print(pred_composition_probs)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0

In [170]:
batch.atom_types = batch.atom_types.to('cuda:0')

In [171]:
atom_type_probs = (
    F.one_hot(batch.atom_types[[range(8)]] - 1, num_classes=MAX_ATOMIC_NUM) +
    pred_composition_probs)

In [172]:
rand_atom_types = torch.multinomial(
    atom_type_probs, num_samples=1).squeeze(1) + 1

In [173]:
rand_atom_types

tensor([31, 31, 31, 31, 52, 52, 31, 31], device='cuda:0')

In [174]:
batch.frac_coords = batch.frac_coords.to('cuda:0')
batch.num_atoms = batch.num_atoms.to('cuda:0')
batch.lengths = batch.lengths.to('cuda:0')
batch.angles = batch.angles.to('cuda:0')

In [175]:
(pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
        pred_composition_per_atom) = self.decode_stats(
            z, batch.num_atoms[[0]], batch.lengths[[0]], batch.angles[[0]], True, gt_elements)

In [176]:
cart_noises_per_atom = (
    torch.randn_like(batch.frac_coords[[8]]))
cart_coords = frac_to_cart_coords(
    batch.frac_coords[[8]], pred_lengths[[0]], pred_angles[[0]], batch.num_atoms[[0]])
cart_coords = cart_coords + cart_noises_per_atom
noisy_frac_coords = cart_to_frac_coords(
    cart_coords[[0]], pred_lengths[[0]], pred_angles[[0]], batch.num_atoms[[0]])

In [177]:
pred_cart_coord_diff, pred_atom_types = self.decoder(
    z, noisy_frac_coords[[0]], rand_atom_types[[range(8)]], batch.num_atoms[[0]], pred_lengths[[0]], pred_angles[[0]], gt_elements[[0]])

In [178]:
print(pred_atom_types)

tensor([[  0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,  -7.0912,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,  -7.7928,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0

The resultant vector should have extremely large negative numbers in all places where the value doesn't correspond to the correct elements 

Second stage: creating a function and verifying the function 

In [179]:
def composition_constraint(self, atom_types, num_atoms, composition_per_atom):
    """
    Restrict the probability distribution from which the atom types are randomly drawn 
    to only include the elements that are present in the crystal.

    atom_types: the atom types in the crystal
    num_atoms: the number of atoms in the crystal
    composition_per_atom: the probability distribution from which the atom types are randomly drawn 

    """

    # Create a range tensor and repeat it as before
    range_tensor = torch.arange(len(num_atoms), device='cuda:0')
    crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

    # Convert atom_types into a mask
    atom_mask = atom_types != 0

    # For each unique crystal_id, get its corresponding indices in composition_per_atom
    unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)

    composition_per_atom = composition_per_atom + 1

    start_idx = 0
    for u_id, count in zip(unique_crystal_ids, counts):
        relevant_elements = atom_types[u_id][atom_mask[u_id]]
        mask = torch.ones_like(composition_per_atom[start_idx])
        mask *= (-10**6)
        mask[relevant_elements-1] = 1

        # Create additive mask
        additive_mask_for_normalization = mask 
        additive_mask_for_normalization[relevant_elements-1] *= 0.0001

        # Apply masks to the relevant segment of composition_per_atom
        composition_per_atom[start_idx:start_idx+count] *= mask
        composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

        # Update start index for next iteration
        start_idx += count
    
    return composition_per_atom

In [180]:
gt_num_atoms = num_atoms
num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(
            z, gt_num_atoms, gt_elements=gt_elements_input)
composition_per_atom = composition_per_atom.to('cuda:0') # this can be removed in native code 
num_atoms = batch.num_atoms[[0]].to('cuda:0')  #this can be removed in native code 

composition_constraint(model, atom_types, num_atoms, composition_per_atom)

tensor([[-2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
          3.6399e-04, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06,  3.2929e-04, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06, -2.0000e+06,
         -2.0000e+06, -2.0000e+06, -2.

Third stage: ensuring that the function works in all contexts in which it is applied. 

Because I validated the performance of the code in a workflow modeled after the forward pass, and the other situations in which the code would be used (langevin dynamics) are highly similar to the forward pass (and I would modularize it into the decode states and decode functions themselves), I don't think further notebook code is neccesary for this third stage in this particular case. 

However, I will take the following steps to verify performance "in-vivo": 
1. For the first few steps / epochs, I will have the code print out the results from the composition constraint ("before/after") that kind of thing. I will do this in the decoder stats function and before and after the decode call at the end of the forward pass. For the langevin dynamics, I will just print it out for every batch because there's less of a risk of slowing things down or taking up memory. After verifying, I'll save the outputs for posterity and get rid of the print statements. 

In [6]:
model.to('cuda:0')

In [42]:
for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)

batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

#wrecking the batch value to demonstrate decoding only from the xrd info
_, _, z = model.encode(None, xrd_int, xrd_loc, atom_spec)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35


In [43]:
self = model

In [44]:
#get the results for the first ten crystals

num_atoms = batch.num_atoms[[range(10)]].to('cuda:0')
atom_spec = atom_spec[[range(10)]].to('cuda:0')
z = z[[range(10)]].to('cuda:0')

In [45]:
print('z is {}'.format(z))
print('atom_spec is {}'.format(atom_spec))
print('num_atoms is {}'.format(num_atoms))

z is tensor([[9.6002e+00, 1.9269e+01, 2.4866e+01, 2.5339e+01, 2.6711e+01, 2.8862e+01,
         2.9078e+01, 3.1650e+01, 3.4939e+01, 3.8625e+01, 3.9111e+01, 4.2631e+01,
         4.3791e+01, 4.4938e+01, 4.6904e+01, 4.8252e+01, 4.9467e+01, 5.1011e+01,
         5.1269e+01, 5.1413e+01, 5.2036e+01, 5.3298e+01, 5.3428e+01, 5.5030e+01,
         5.6139e+01, 5.7205e+01, 5.9793e+01, 6.0145e+01, 6.0275e+01, 6.1076e+01,
         6.2767e+01, 6.6103e+01, 6.6226e+01, 6.8176e+01, 6.9449e+01, 6.9449e+01,
         6.9663e+01, 6.9663e+01, 6.9783e+01, 7.0304e+01, 7.0304e+01, 7.1366e+01,
         7.1366e+01, 7.1603e+01, 7.1713e+01, 7.2841e+01, 7.2841e+01, 7.3796e+01,
         7.4721e+01, 7.4721e+01, 7.6997e+01, 7.6997e+01, 7.7228e+01, 7.7425e+01,
         7.8139e+01, 7.9660e+01, 7.9660e+01, 8.0467e+01, 8.0467e+01, 8.1279e+01,
         8.1279e+01, 8.2706e+01, 8.2706e+01, 8.2819e+01, 8.3134e+01, 8.3708e+01,
         8.3708e+01, 8.4047e+01, 8.6133e+01, 8.6133e+01, 8.7732e+01, 8.7732e+01,
         8.7852e+01, 8.

In [46]:
#number of steps set unusually low to get a rough estimate of performance
ld_kwargs = SimpleNamespace(n_step_each=10,
                                step_lr=1e-4,
                                min_sigma=0,
                                save_traj=False,
                                disable_bar=False)

force_num_atoms = True
gt_num_atoms = num_atoms if force_num_atoms else None

gt_atom_types = None

outputs = model.langevin_dynamics(
    z, ld_kwargs, gt_num_atoms, gt_atom_types, atom_spec)

100%|██████████| 50/50 [00:10<00:00,  4.90it/s]


Below are the predicted outputs (based on the 20% accuracy, at least one of these should be reasonable) 

In [47]:
batch

Batch(edge_index=[2, 19564], y=[256, 1], frac_coords=[2651, 3], atom_types=[2651], lengths=[256, 3], angles=[256, 3], to_jimages=[19564, 3], num_atoms=[256], num_bonds=[256], num_nodes=2651, batch=[2651], ptr=[257])

In [48]:
batch.lengths[[range(10)]]

tensor([[ 4.1346,  4.1346, 18.4256],
        [ 3.7172,  3.7172,  6.3843],
        [ 2.4475,  2.4475,  4.1407],
        [ 3.4297,  5.9062,  7.2953],
        [ 5.1351,  5.1351,  8.3894],
        [ 5.1473,  5.2151,  5.6750],
        [ 3.3229,  7.3325,  7.3955],
        [ 2.9694,  2.9694, 10.9180],
        [ 4.7658,  6.5318,  6.5318],
        [ 4.1572,  5.3982,  7.1205]])

In [49]:
outputs['lengths']

tensor([[ 4.0205,  5.1997, 11.2137],
        [ 3.5160,  3.4647,  6.8485],
        [ 1.9670,  2.1898,  4.8860],
        [ 3.6342,  5.1444,  7.4281],
        [ 5.4465,  6.1498,  8.9752],
        [ 4.2128,  5.5710,  6.2468],
        [ 4.8880,  6.7042,  8.3586],
        [ 2.2711,  4.4781,  9.3519],
        [ 4.7696,  6.6597,  6.1514],
        [ 4.0176,  6.9651,  9.6162]], device='cuda:0')

In [28]:
print(batch.lengths[[3]])
print(batch.num_atoms[[0,1,2,3]])
print(batch.frac_coords[[range(8+4+2,8+4+2+7)]])

tensor([[3.4297, 5.9062, 7.2953]])
tensor([8, 4, 2, 7])
tensor([[0.7567, 0.9136, 0.5133],
        [0.5069, 0.4940, 0.0137],
        [0.0070, 0.0101, 0.0140],
        [0.1071, 0.7052, 0.2142],
        [0.6085, 0.2007, 0.2170],
        [0.4078, 0.8065, 0.8157],
        [0.9050, 0.3029, 0.8100]])


In [29]:
outputs['lengths'][[3]]
outputs['frac_coords'][[range(8+4+2,8+4+2+7)]]

tensor([[0.7775, 0.2084, 0.3568],
        [0.2598, 0.9367, 0.0430],
        [0.7804, 0.7123, 0.8610],
        [0.2773, 0.2111, 0.8565],
        [0.2948, 0.4820, 0.6685],
        [0.7887, 0.9983, 0.6970],
        [0.7668, 0.4253, 0.0138]], device='cuda:0')

Below are the ground truth inputs